# Applied A/B Testing - Udacity Experiment Analysis

## Executive Summary

This notebook provides a deep-dive analysis of an A/B test conducted by Udacity. The experiment evaluates the impact of a landing page redesign on user conversion rates across a population of nearly 300,000 users. 

### Analytical Framework:
1. **Data Integrity**: Cleansing mismatches between group assignment and page delivery.
2. **Global Comparison**: Analyzing baseline vs. treatment performance.
3. **Statistical Robustness**: Implementing Z-tests for proportions and calculating 95% Confidence Intervals.
4. **Post-hoc Power Analysis**: Evaluating the experiment's ability to detect the observed difference.
5. **Heterogeneity Analysis**: Investigating segment-level significance to identify potential local gains.

## 1. Environment and Data Acquisition

Initialization of the analytical stack and loading the primary experimental dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import TTestIndPower
import warnings
warnings.filterwarnings('ignore')

# Dataset path (adjust if necessary)
path = '../data/ab_data.csv'
df = pd.read_csv(path)

print(f"Initial dataset size: {len(df):,} records.")

## 2. Data Integrity and Cleansing

In large-scale experimentation, deployment errors can lead to a subset of users seeing a version inconsistent with their assigned group (e.g., control users seeing the new page). 

**Objective**: Remove these records to maintain a clean intent-to-treat (ITT) analysis.

In [ ]:
# Filter for consistent assignments
df_clean = df[
    ((df["group"] == "treatment") & (df["landing_page"] == "new_page")) |
    ((df["group"] == "control") & (df["landing_page"] == "old_page"))
]

removed_count = len(df) - len(df_clean)
print(f"Records removed due to inconsistency: {removed_count:,}")
print(f"Final analytical dataset: {len(df_clean):,} records.")

# Ensure binary conversion
df_clean["treatment"] = (df_clean["group"] == "treatment").astype(int)
df_clean["conversion"] = df_clean["converted"].astype(int)

## 3. Global Performance Metrics

Calculating the observed conversion rates (CR) and the absolute/relative uplift.

In [ ]:
control_group = df_clean[df_clean["treatment"] == 0]["conversion"]
treatment_group = df_clean[df_clean["treatment"] == 1]["conversion"]

n_control, n_treat = len(control_group), len(treatment_group)
conv_control, conv_treat = control_group.sum(), treatment_group.sum()

cr_control = conv_control / n_control
cr_treat = conv_treat / n_treat

uplift_abs = cr_treat - cr_control
uplift_rel = (cr_treat - cr_control) / cr_control

print(f"Control CR: {cr_control:.4%}")
print(f"Treatment CR: {cr_treat:.4%}")
print(f"Absolute Uplift: {uplift_abs:+.4%}")
print(f"Relative Uplift: {uplift_rel:+.2%}")

## 4. Inferential Statistical Validation

We use a **Two-Sample Z-Test for Proportions**, which is more statistically appropriate than a T-test for binary outcomes with large sample sizes. We also calculate 95% Confidence Intervals (CI) to quantify uncertainty.

In [ ]:
# Z-test for proportions
count = np.array([conv_treat, conv_control])
nobs = np.array([n_treat, n_control])

z_stat, p_val = proportions_ztest(count, nobs, alternative='two-sided')

# Confidence Intervals
(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(count, nobs, alpha=0.05, method='normal')

print(f"Z-Statistic: {z_stat:.4f}")
print(f"P-Value: {p_val:.6f}")
print(f"\nControl 95% CI: [{lower_treat:.4%}, {upper_treat:.4%}]")
print(f"Treatment 95% CI: [{lower_con:.4%}, {upper_con:.4%}]")

## 5. Post-hoc Power Analysis

Given that the result is non-significant, we calculate the **Statistical Power** to understand if the sample size was sufficient to detect a small but meaningful effect (e.g., a 1% relative improvement).

In [ ]:
analysis = TTestIndPower()
pooled_std = np.sqrt(cr_control * (1 - cr_control))
effect_size = (cr_control * 0.01) / pooled_std # Assuming MDE of 1% relative

power = analysis.solve_power(effect_size=effect_size, nobs1=n_control, alpha=0.05, ratio=1.0)
print(f"Statistical Power to detect a 1% relative uplift: {power:.2%}")

## 6. Heterogeneity Analysis (Segmentation)

Investigating if the treatment effect varies across visit segments (Day vs. Night). We run separate Z-tests for each segment.

In [ ]:
df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])
df_clean["hour"] = df_clean["timestamp"].dt.hour
df_clean["time_segment"] = df_clean["hour"].apply(lambda x: "day" if 8 <= x <= 20 else "night")

for segment in ["day", "night"]:
    seg_df = df_clean[df_clean["time_segment"] == segment]
    
    s_control = seg_df[seg_df["treatment"] == 0]["conversion"]
    s_treat = seg_df[seg_df["treatment"] == 1]["conversion"]
    
    s_count = np.array([s_treat.sum(), s_control.sum()])
    s_nobs = np.array([len(s_treat), len(s_control)])
    
    _, s_p = proportions_ztest(s_count, s_nobs)
    
    print(f"--- Segment: {segment.upper()} ---")
    print(f"Control CR: {s_control.mean():.4%}")
    print(f"Treatment CR: {s_treat.mean():.4%}")
    print(f"P-Value: {s_p:.6f}")
    print()

## 7. Professional Visualization

Comparing the distributions of the estimated conversion rates for both groups.

In [ ]:
x = np.linspace(0.115, 0.125, 1000)
y_control = stats.norm.pdf(x, cr_control, np.sqrt(cr_control*(1-cr_control)/n_control))
y_treat = stats.norm.pdf(x, cr_treat, np.sqrt(cr_treat*(1-cr_treat)/n_treat))

plt.figure(figsize=(10, 6))
plt.plot(x, y_control, label='Control (Old Page)', color='#e74c3c', lw=2)
plt.plot(x, y_treat, label='Treatment (New Page)', color='#2ecc71', lw=2)
plt.axvline(cr_control, color='#e74c3c', linestyle='--')
plt.axvline(cr_treat, color='#2ecc71', linestyle='--')

plt.title('Estimated Conversion Rate Distributions', fontsize=14)
plt.xlabel('Conversion Rate', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Technical Conclusions and Business Recommendations

### Core Findings
1. **No Evidence of Improvement**: The treatment version (New Page) achieved a conversion rate of **11.88%**, slightly lower than the control's **12.04%**. The resulting relative uplift of **-1.31%** indicates underperformance.
2. **Statistical Non-Significance**: With a p-value of **0.1897** (exceeding the 0.05 threshold), we fail to reject the null hypothesis. The observed difference is consistent with random sampling noise.
3. **High Overlap**: As visualized in the distribution density plot, the confidence intervals for both groups significantly overlap, confirming the lack of a distinct performance gap.
4. **Segment Stability**: The negative trend was consistent across both day and night segments, ruling out the possibility of time-based heterogeneous gains.

### Business Recommendation
**Do not launch the new landing page.** Deploying the redesign would likely maintain the status quo or cause a marginal decrease in conversions. The experiment had high statistical power (>90%) to detect even small changes, making the current negative result very reliable. 

**Next Action**: Conduct a qualitative post-mortem or heuristic evaluation to identify why the redesign failed to engage users as hypothesized, and iterate on a new design concept.